In [3]:
import pandas as pd
import numpy as np

#Load the dataset
df = pd.read_csv('Project1_Retail_Sales_50K_Rows_8_Columns.csv')

In [4]:
#Inspect shape, columns, and data types
print("--- Dataset Info ---")
print(f"Total Rows & Columns: {df.shape}")
print(df.dtypes)

--- Dataset Info ---
Total Rows & Columns: (50000, 8)
OrderDate        object
Region           object
Category         object
ProductName      object
Quantity          int64
UnitPriceINR    float64
DiscountRate    float64
UnitCostINR     float64
dtype: object


In [5]:
#Check for missing values or weird duplicates
print("\n--- Missing Values ---")
print(df.isnull().sum())


--- Missing Values ---
OrderDate       0
Region          0
Category        0
ProductName     0
Quantity        0
UnitPriceINR    0
DiscountRate    0
UnitCostINR     0
dtype: int64


In [6]:
#Data Validation: Convert OrderDate to datetime and check value logic
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
print("\n--- Validation Check ---")
print("Negative Quantities:", (df['Quantity'] < 0).sum())
print("Discounts out of bounds (<0 or >1):", ((df['DiscountRate'] < 0) | (df['DiscountRate'] > 1)).sum())


--- Validation Check ---
Negative Quantities: 0
Discounts out of bounds (<0 or >1): 0


In [7]:
#Feature Engineering (Calculating Financial KPIs)
#LineRevenue = Quantity * UnitPrice * (1 - Discount)
df['LineRevenue'] = df['Quantity'] * df['UnitPriceINR'] * (1 - df['DiscountRate'])
df['LineCost'] = df['Quantity'] * df['UnitCostINR']
df['LineProfit'] = df['LineRevenue'] - df['LineCost']
df['ProfitMargin'] = df['LineProfit'] / df['LineRevenue']

In [8]:
print("\n--- Enriched Data Preview ---")
display(df.head())


--- Enriched Data Preview ---


,OrderDate,Region,Category,ProductName,Quantity,UnitPriceINR,DiscountRate,UnitCostINR,LineRevenue,LineCost,LineProfit,ProfitMargin
0,2025-02-04,North,Technology,Mouse,1,950.0,0.15,420.0,807.5,420.0,387.5,0.479876
1,2024-08-18,West,Home & Living,Storage Container Set,2,890.0,0.05,390.0,1691.0,780.0,911.0,0.538734
2,2024-11-08,West,Technology,Laptop,3,52000.0,0.00,34500.0,156000.0,103500.0,52500.0,0.336538
3,2025-03-12,East,Technology,USB-C Hub,4,2600.0,0.15,1250.0,8840.0,5000.0,3840.0,0.434389
4,2025-09-28,West,Furniture,Work Desk,2,7200.0,0.10,2850.0,12960.0,5700.0,7260.0,0.560185


In [ ]:
#SQL lite

In [9]:
import sqlite3
import pandas as pd
import random

#Connect to a local SQLite file database
conn = sqlite3.connect('retail_capstone.db')
cursor = conn.cursor()

# Drop tables if they already exist to ensure a clean slate
cursor.execute('DROP TABLE IF EXISTS Orders;')
cursor.execute('DROP TABLE IF EXISTS Products;')
cursor.execute('DROP TABLE IF EXISTS Stores;')
cursor.execute('DROP TABLE IF EXISTS Customers;')

#Create Normalized Schema Tables (Products, Stores, Customers, Orders)
cursor.execute('''
CREATE TABLE Products (
    ProductID INTEGER PRIMARY KEY AUTOINCREMENT,
    ProductName TEXT UNIQUE,
    Category TEXT
);
''')

cursor.execute('''
CREATE TABLE Stores (
    StoreID INTEGER PRIMARY KEY AUTOINCREMENT,
    StoreName TEXT,
    Region TEXT
);
''')

cursor.execute('''
CREATE TABLE Customers (
    CustomerID INTEGER PRIMARY KEY AUTOINCREMENT,
    Segment TEXT,
    Region TEXT
);
''')

cursor.execute('''
CREATE TABLE Orders (
    OrderID INTEGER PRIMARY KEY AUTOINCREMENT,
    OrderDate TEXT,
    CustomerID INTEGER,
    ProductID INTEGER,
    StoreID INTEGER,
    Quantity INTEGER,
    UnitPrice REAL,
    DiscountRate REAL,
    UnitCost REAL,
    FOREIGN KEY(ProductID) REFERENCES Products(ProductID),
    FOREIGN KEY(StoreID) REFERENCES Stores(StoreID),
    FOREIGN KEY(CustomerID) REFERENCES Customers(CustomerID)
);
''')

#Populate Dimension Tables from our dataset
products_df = df[['ProductName', 'Category']].drop_duplicates().reset_index(drop=True)
# Use if_exists='append' so SQL schema containing AUTOINCREMENT ProductID is preserved
products_df.to_sql('Products', conn, if_exists='append', index=False)

stores_data = [{'StoreName': f'Store {reg}', 'Region': reg} for reg in df['Region'].unique()]
pd.DataFrame(stores_data).to_sql('Stores', conn, if_exists='append', index=False)

random.seed(42)
segments = ['Consumer', 'Corporate', 'Home Office']
customers_data = [{'Segment': random.choice(segments), 'Region': random.choice(df['Region'].unique())} for _ in range(1, 501)]
pd.DataFrame(customers_data).to_sql('Customers', conn, if_exists='append', index=False)

#Populate the Fact Table (Orders) with Foreign Keys
prod_map = pd.read_sql('SELECT ProductID, ProductName FROM Products', conn).set_index('ProductName')['ProductID'].to_dict()
store_map = pd.read_sql('SELECT StoreID, Region FROM Stores', conn).set_index('Region')['StoreID'].to_dict()
cust_ids = pd.read_sql('SELECT CustomerID FROM Customers', conn)['CustomerID'].tolist()

orders_data = []
for idx, row in df.iterrows():
    orders_data.append({
        'OrderDate': str(row['OrderDate']),
        'CustomerID': random.choice(cust_ids),
        'ProductID': prod_map[row['ProductName']],
        'StoreID': store_map[row['Region']],
        'Quantity': row['Quantity'],
        'UnitPrice': row['UnitPriceINR'],
        'DiscountRate': row['DiscountRate'],
        'UnitCost': row['UnitCostINR']
    })

pd.DataFrame(orders_data).to_sql('Orders', conn, if_exists='append', index=False)

# Create performance indexes
cursor.execute('CREATE INDEX IF NOT EXISTS idx_orders_date ON Orders(OrderDate);')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_orders_prod ON Orders(ProductID);')
conn.commit()
print("--- SQLite Database Tables Created & Populated Successfully! ---")

--- SQLite Database Tables Created & Populated Successfully! ---


In [10]:
# Multi-step SQL query using JOINs and CTEs to extract Category Performance
sql_query = """
WITH CalculatedOrders AS (
    SELECT
        p.Category,
        (o.Quantity * o.UnitPrice * (1 - o.DiscountRate)) AS LineRevenue,
        (o.Quantity * o.UnitCost) AS LineCost
    FROM Orders o
    JOIN Products p ON o.ProductID = p.ProductID
)
SELECT
    Category,
    ROUND(SUM(LineRevenue), 2) AS TotalRevenue,
    ROUND(SUM(LineRevenue - LineCost), 2) AS TotalProfit,
    ROUND(SUM(LineRevenue - LineCost) / NULLIF(SUM(LineRevenue), 0) * 100, 2) AS ProfitMarginPercentage
FROM CalculatedOrders
GROUP BY Category
ORDER BY TotalRevenue DESC;
"""

category_sql_results = pd.read_sql(sql_query, conn)
print("--- SQL Query Results ---")
display(category_sql_results)

# Close connection
conn.close()

--- SQL Query Results ---


,Category,TotalRevenue,TotalProfit,ProfitMarginPercentage
0,Technology,734735155.0,223538795.0,30.42
1,Furniture,169425550.0,95296100.0,56.25
2,Home & Living,53899050.0,23069750.0,42.80
3,Office Supplies,13782588.5,6772668.5,49.14


In [11]:
from google.colab import files

# 1. Save your cleaned dataframe to a CSV file locally in Colab
df.to_csv('Cleaned_Retail_Sales.csv', index=False)
files.download('Cleaned_Retail_Sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>